# Verify Minimal Architecure

This notebook uses the sage backend to evaluate whether or not a given architecture with given exponent is genuinely a minimal filling architecture.

## Load Functions

In [1]:
from dim_backprop_gpu_only import compute_dimension
from is_unimodal import is_unimodal

c:\Users\daoke\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\cupy\_environment.py:284: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


## Functions to Verify Minimality

In [2]:
def check_full_dimension_fast(sizes, exponent=2):
    """
    Computes the functional dimension by wrapping the existing compute_dimension 
    function from dim_backprop.py.
    
    Returns:
        tuple: (is_filling, actual_dimension, ambient_dimension, num_params)
    """
    # compute_dimension returns: (sizes, exponent, ambient_dim, expected_dim, actual_dim, defect)
    result = compute_dimension(sizes, exponent)
    
    ambient_dim = result[2]
    actual_dim = result[4]
    
    # Calculate num_params (since compute_dimension doesn't explicitly return it)
    num_params = sum([m * n for m, n in zip(sizes[:-1], sizes[1:])])
    
    # If parameters < ambient_dim, it trivially can't fill
    if num_params < ambient_dim:
        return False, actual_dim, ambient_dim, num_params
        
    # The architecture fills the space if its actual dimension equals the ambient dimension
    is_filling = (actual_dim == ambient_dim)
    
    return is_filling, actual_dim, ambient_dim, num_params

In [3]:
def verify_minimality_brute_force(hidden_tuple, d_0, d_h, exponent=2):

    """
    Checks if a given architecture (d_0, hidden_tuple, d_h) is a minimal filling architecture
    by checking immediate subarchitectures (subtracting 1 from each coordinate).
    """

    target_sizes = [d_0] + list(hidden_tuple) + [d_h]
    print(f"="*100)
    print(f"EVALUATING TARGET ARCHITECTURE: {target_sizes}")
    print("="*100)
    
    # 1. Check if the target is FILLING
    is_full, dim, amb, params = check_full_dimension_fast(target_sizes, exponent)
    status = "FILLING" if is_full else "SHORT"
    print(f"\nTarget Result: [{status}] (Rank: {dim}/{amb}, Params: {params})")
    
    if not is_full:
        print("\n>>> CONCLUSION: NOT MINIMAL FILLING (Target itself does not fill the dimension) <<<")
        return False
    
    # 2. Generate immediate subarchitectures (subtract 1 from each coordinate)
    print("\nTarget is FILLING! Generating immediate smaller subsets to test:\n")
    
    immediate_subsets = []
    for i in range(len(hidden_tuple)):
        # Only subtract 1 if the layer width is greater than 1 
        # a width of 0 should break
        if hidden_tuple[i] > 1:
            subset = list(hidden_tuple)
            subset[i] -= 1
            immediate_subsets.append(tuple(subset))
    
    print(f"Total immediate subsets to verify: {len(immediate_subsets)}\n")
    
    # 3. Check the immediate predecessors
    for subset in immediate_subsets:
        pred_sizes = [d_0] + list(subset) + [d_h]
        
        pred_is_full, pred_dim, pred_amb, pred_params = check_full_dimension_fast(pred_sizes, exponent)
        
        p_status = "FILLING" if pred_is_full else "SHORT"
        print(f"  Testing Subset {pred_sizes} -> [{p_status}] (Jacobian Rank: {pred_dim}/{pred_amb}, Params: {pred_params})")
        
        if pred_is_full:
            print(f"\n>>> CONCLUSION: IT IS NOT A MINIMAL FILLING ARCHITECTURE <<<")
            print(f"A strictly smaller architecture {pred_sizes} is also FULL.")
            return False
            
    print(f"\n>>> CONCLUSION: CONFIRMED MINIMAL FILLING ARCHITECTURE <<<")
    print(f"The architecture is MINIMAL FILLING, and no immediate subarchitecture can fill the space.")
    return True

# Examples

In [4]:
d_0,d_h = 2,1
verify_minimality_brute_force([3, 3, 4, 7, 6, 8, 4], d_0, d_h, exponent=2)

EVALUATING TARGET ARCHITECTURE: [2, 3, 3, 4, 7, 6, 8, 4, 1]

Target Result: [FILLING] (Rank: 129/129, Params: 181)

Target is FILLING! Generating immediate smaller subsets to test:

Total immediate subsets to verify: 7

  Testing Subset [2, 2, 3, 4, 7, 6, 8, 4, 1] -> [SHORT] (Jacobian Rank: 67/129, Params: 176)
  Testing Subset [2, 3, 2, 4, 7, 6, 8, 4, 1] -> [SHORT] (Jacobian Rank: 37/129, Params: 174)
  Testing Subset [2, 3, 3, 3, 7, 6, 8, 4, 1] -> [SHORT] (Jacobian Rank: 108/129, Params: 171)
  Testing Subset [2, 3, 3, 4, 6, 6, 8, 4, 1] -> [SHORT] (Jacobian Rank: 128/129, Params: 171)
  Testing Subset [2, 3, 3, 4, 7, 5, 8, 4, 1] -> [SHORT] (Jacobian Rank: 113/129, Params: 166)
  Testing Subset [2, 3, 3, 4, 7, 6, 7, 4, 1] -> [SHORT] (Jacobian Rank: 125/129, Params: 171)
  Testing Subset [2, 3, 3, 4, 7, 6, 8, 3, 1] -> [SHORT] (Jacobian Rank: 127/129, Params: 172)

>>> CONCLUSION: CONFIRMED MINIMAL FILLING ARCHITECTURE <<<
The architecture is MINIMAL FILLING, and no immediate subarchite

True

In [5]:
d_0,d_h = 2,1
verify_minimality_brute_force([3, 4, 4, 4, 7, 10, 3], d_0, d_h, exponent=2)

EVALUATING TARGET ARCHITECTURE: [2, 3, 4, 4, 4, 7, 10, 3, 1]

Target Result: [FILLING] (Rank: 129/129, Params: 181)

Target is FILLING! Generating immediate smaller subsets to test:

Total immediate subsets to verify: 7

  Testing Subset [2, 2, 4, 4, 4, 7, 10, 3, 1] -> [SHORT] (Jacobian Rank: 67/129, Params: 175)
  Testing Subset [2, 3, 3, 4, 4, 7, 10, 3, 1] -> [SHORT] (Jacobian Rank: 128/129, Params: 174)
  Testing Subset [2, 3, 4, 3, 4, 7, 10, 3, 1] -> [SHORT] (Jacobian Rank: 119/129, Params: 173)
  Testing Subset [2, 3, 4, 4, 3, 7, 10, 3, 1] -> [SHORT] (Jacobian Rank: 66/129, Params: 170)
  Testing Subset [2, 3, 4, 4, 4, 6, 10, 3, 1] -> [SHORT] (Jacobian Rank: 106/129, Params: 167)
  Testing Subset [2, 3, 4, 4, 4, 7, 9, 3, 1] -> [SHORT] (Jacobian Rank: 127/129, Params: 171)
  Testing Subset [2, 3, 4, 4, 4, 7, 10, 2, 1] -> [SHORT] (Jacobian Rank: 97/129, Params: 170)

>>> CONCLUSION: CONFIRMED MINIMAL FILLING ARCHITECTURE <<<
The architecture is MINIMAL FILLING, and no immediate suba

True

In [ ]:
# # 5-modal
# d_0,d_h = 2,1
# verify_minimality_brute_force([3,4,5,4,7,6,14,9,36,29,67,11], d_0, d_h, exponent=2)

In [ ]:
# # 6-modal
# d_0,d_h = 2,1
# verify_minimality_brute_force([3,4,6,5,11,9,35,25,100,39,143,13,50,40], d_0, d_h, exponent=2)

# Update .csv to Correctly Identify is_minimal Tuples

In [ ]:
import ast
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/raw")
REQUIRED = {"h", "architecture", "exponent", "is_minimal"}

for csv_path in DATA_DIR.glob("*.csv"):
    try:
        df = pd.read_csv(csv_path)
    except (pd.errors.EmptyDataError, pd.errors.ParserError):
        continue

    if not REQUIRED.issubset(df.columns):
        continue

    df["is_unimodal"] = df["architecture"].apply(is_unimodal)

    for index, row in df[df["is_minimal"] == True].iterrows():
        arch = ast.literal_eval(row["architecture"]) if isinstance(
            row["architecture"], str
        ) else row["architecture"]

        if not verify_minimality_brute_force(
            hidden_tuple=tuple(arch[1:-1]),
            d_0=arch[0],
            d_h=arch[-1],
            exponent=int(row["exponent"]),
        ):
            df.at[index, "is_minimal"] = False

    df.to_csv(csv_path, index=False)